# 🔤 Thai Character Recognition — Transfer Learning

จำแนกตัวอักษร/ตัวเลข/วรรณยุกต์ไทย **72 คลาส** ด้วย Transfer Learning
เปรียบเทียบ 3 โมเดล: **ResNet50**, **EfficientNet-B3**, **MobileNetV3-Large**

โค้ดหลักอยู่ในแพ็กเกจ `src/` notebook นี้เป็นตัวเรียกใช้ (orchestrator):

| Module | หน้าที่ |
|---|---|
| `src/dataset.py` | transforms, stratified split, `ThaiCharDataset`, อ่าน `class_mapping.json` |
| `src/model.py` | `create_model()` (grayscale 1-ch), `get_device()` |
| `src/train.py` | `train_model()`, `evaluate()`, `run()` + CLI |
| `src/predict.py` | `test_single_image()` — inference ที่ pipeline ตรงกับตอนเทรน |

## ⚙️ วิธีใช้งาน

1. จัด dataset เป็น `<class_name>/<image>.jpg` (เช่น `kor_kai/0001.jpg`) ไว้ใน root เดียวกับ notebook
   — รายชื่อ 72 คลาสดูได้จาก `class_mapping.json`
2. ติดตั้ง dependencies: `pip install -r requirements.txt`
3. **เทรน** (ใช้เวลานานบน CPU/MPS) รันจาก terminal:
   ```bash
   python -m src.train --models all --epochs 10 --augment single
   ```
4. รัน notebook นี้เพื่อดูผล/ประเมิน/ทำนายภาพเดียว

In [ ]:
import os, json
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.model import create_model, get_device, MODEL_LABELS, SUPPORTED_MODELS
from src.dataset import (build_samples, stratified_split, load_class_mapping,
                         ThaiCharDataset, val_test_transforms, _TRAIN_AUGMENTS)
from src.train import OUTPUT_DIRS, evaluate, run as train_run
from src.predict import test_single_image, load_trained_model

DATA_ROOT = '.'                       # โฟลเดอร์คลาสอยู่ข้าง notebook
MAPPING_PATH = 'class_mapping.json'
DEVICE = get_device()
print('PyTorch', torch.__version__, '| device:', DEVICE)

In [ ]:
# ตั้งค่าฟอนต์ไทยสำหรับ matplotlib (cross-platform: macOS / Windows / Linux)
import matplotlib.font_manager as fm
from matplotlib import rcParams

def setup_thai_font():
    thai_fonts = [
        'Thonburi', 'Sukhumvit Set', 'Ayuthaya',        # macOS
        'Tahoma', 'Leelawadee UI', 'Angsana New',       # Windows
        'TH Sarabun New', 'Noto Sans Thai', 'Loma',     # Linux / ทั่วไป
    ]
    available = {f.name for f in fm.fontManager.ttflist}
    selected = next((f for f in thai_fonts if f in available), 'DejaVu Sans')
    rcParams['font.family'] = selected
    rcParams['font.sans-serif'] = [selected] + rcParams['font.sans-serif']
    rcParams['axes.unicode_minus'] = False
    return selected

print('✅ ฟอนต์:', setup_thai_font())

## 1. Dataset & Stratified Split

โหลดรายชื่อภาพจาก 72 โฟลเดอร์ที่ระบุใน `class_mapping.json` เท่านั้น
(ไม่หลงไปสแกน `outputs_*/`, `src/`) แล้วแบ่ง train/val/test แบบ **stratified per-class**
เพื่อให้ทุกคลาสมีตัวแทนในทุก split — รวมคลาสที่ภาพน้อย

In [ ]:
samples, class_names, class_to_idx, idx_to_char = build_samples(DATA_ROOT, MAPPING_PATH)
num_classes = len(class_names)
print(f'รวม {len(samples)} ภาพ | {num_classes} คลาส')

# นับจำนวนภาพต่อคลาส
from collections import Counter
counts = Counter(lbl for _, lbl in samples)
per_class = sorted(((class_names[i], counts[i]) for i in range(num_classes)),
                   key=lambda x: x[1])
print('คลาสที่ภาพน้อยสุด 5 อันดับ:', per_class[:5])
print('คลาสที่ภาพมากสุด 5 อันดับ:', per_class[-5:])

train_idx, val_idx, test_idx = stratified_split(samples, seed=42)
print(f'\ntrain={len(train_idx)}  val={len(val_idx)}  test={len(test_idx)}')

## 2. Augmentation Preview

Training augment 3 ระดับ (gentle / mild / strong) — โหมด `single` สุ่ม 1 ระดับต่อภาพ,
โหมด `triple` ใช้ครบทั้ง 3 (ดาต้า ×3) ทั้งหมดเป็น grayscale + `Normalize((0.5,),(0.5,))`

In [ ]:
from PIL import Image

def denorm(t):
    return torch.clamp(t * 0.5 + 0.5, 0, 1).squeeze(0).numpy()

path, label = samples[train_idx[0]]
orig = Image.open(path).convert('L')
titles = ['Original', 'Gentle', 'Mild', 'Strong']
imgs = [np.array(orig) / 255.0] + [denorm(t(orig)) for t in _TRAIN_AUGMENTS]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, im, ti in zip(axes, imgs, titles):
    ax.imshow(im, cmap='gray'); ax.set_title(ti); ax.axis('off')
fig.suptitle(f"ตัวอย่าง augment — {idx_to_char[label]} ({class_names[label]})", fontsize=14)
plt.tight_layout(); plt.show()

## 3. Training

เทรนใช้เวลานาน — แนะนำรันจาก **terminal** (เห็น progress bar ชัด, ไม่ค้าง kernel):
```bash
python -m src.train --models all --epochs 10 --augment single
```
เซลล์ด้านล่าง guard ไว้ด้วย `RUN_TRAINING = False` เพื่อไม่ให้ Restart&RunAll เผลอเทรนใหม่หลายชั่วโมง
ผลลัพธ์ถูกเซฟไว้ที่ `outputs_<model>/` และสรุปที่ `results_summary.json`

In [ ]:
RUN_TRAINING = False  # ตั้งเป็น True ถ้าต้องการเทรนจาก notebook

if RUN_TRAINING:
    results = train_run(SUPPORTED_MODELS, epochs=10, batch_size=32, lr=1e-3,
                        max_per_class=0, num_workers=0, seed=42, augment='single')
else:
    print('ℹ️ ข้ามการเทรน — โหลดผลจาก outputs_*/ แทน')
    print('   ตั้ง RUN_TRAINING = True หรือรัน: python -m src.train --models all')

## 4. Training Results

โหลด `results_summary.json` และกราฟ loss/accuracy ต่อ epoch ของแต่ละโมเดล

In [ ]:
if os.path.exists('results_summary.json'):
    summary = json.load(open('results_summary.json', encoding='utf-8'))
    print('📊 สรุปผล Test Set')
    print(f"{'Model':<22}{'Test Acc':>10}{'Macro F1':>10}{'Best Val':>10}{'Epochs':>8}")
    for name in SUPPORTED_MODELS:
        if name in summary:
            r = summary[name]
            print(f"{MODEL_LABELS[name]:<22}"
                  f"{r.get('test_accuracy',0)*100:>9.2f}%"
                  f"{r.get('macro_f1',0):>10.4f}"
                  f"{r.get('best_val_acc',0)*100:>9.2f}%"
                  f"{r.get('epochs_run',0):>8}")
else:
    print('⚠️ ยังไม่มี results_summary.json — เทรนก่อน')

In [ ]:
# กราฟ training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for name in SUPPORTED_MODELS:
    hp = os.path.join(OUTPUT_DIRS[name], 'training_history.json')
    if not os.path.exists(hp):
        continue
    h = json.load(open(hp))
    ep = range(1, len(h['val_loss']) + 1)
    axes[0].plot(ep, h['val_loss'], marker='o', label=MODEL_LABELS[name])
    axes[1].plot(ep, [a*100 for a in h['val_acc']], marker='o', label=MODEL_LABELS[name])
axes[0].set_title('Validation Loss'); axes[0].set_xlabel('Epoch'); axes[0].grid(alpha=.3); axes[0].legend()
axes[1].set_title('Validation Accuracy (%)'); axes[1].set_xlabel('Epoch'); axes[1].grid(alpha=.3); axes[1].legend()
plt.tight_layout(); plt.show()

## 5. Evaluation & Confusion Matrix

เลือกโมเดล โหลด best weights แล้วประเมินบน test set + วาด confusion matrix

In [ ]:
EVAL_MODEL = 'efficientnet_b3'  # 'resnet50' | 'efficientnet_b3' | 'mobilenet_v3'

model_path = os.path.join(OUTPUT_DIRS[EVAL_MODEL], f'{EVAL_MODEL}_best.pt')
if os.path.exists(model_path):
    model, _, _ = load_trained_model(EVAL_MODEL, DEVICE)
    test_ds = ThaiCharDataset(samples, test_idx, mode='test')
    test_loader = torch.utils.data.DataLoader(test_ds, batch_size=64, shuffle=False)
    acc, report = evaluate(model, test_loader, class_names, DEVICE)
    print(f'{MODEL_LABELS[EVAL_MODEL]} — Test Accuracy: {acc*100:.2f}%')
    print(f"Macro F1: {report['macro avg']['f1-score']:.4f}")
else:
    print(f'⚠️ ไม่พบ {model_path} — เทรนโมเดลนี้ก่อน')

In [ ]:
# Confusion matrix (เฉพาะเมื่อมีโมเดลแล้ว)
from sklearn.metrics import confusion_matrix
if os.path.exists(model_path):
    model.eval(); preds, labels = [], []
    with torch.no_grad():
        for x, y in test_loader:
            out = model(x.to(DEVICE))
            preds.extend(out.argmax(1).cpu().numpy()); labels.extend(y.numpy())
    uniq = sorted(set(labels))
    cm = confusion_matrix(labels, preds, labels=uniq)
    names = [idx_to_char[i] for i in uniq]
    plt.figure(figsize=(20, 18))
    sns.heatmap(cm, cmap='Blues', square=True, xticklabels=names, yticklabels=names,
                linewidths=.1, linecolor='gray')
    plt.title(f'Confusion Matrix — {MODEL_LABELS[EVAL_MODEL]} (Acc {acc*100:.2f}%)', fontsize=16)
    plt.xlabel('Predicted'); plt.ylabel('True')
    plt.xticks(rotation=90, fontsize=7); plt.yticks(rotation=0, fontsize=7)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIRS[EVAL_MODEL], f'confusion_matrix_{EVAL_MODEL}.png'), dpi=200, bbox_inches='tight')
    plt.show()

## 6. ทำนายภาพเดียว (Single Image Inference)

`test_single_image()` ใช้ transform เดียวกับตอนเทรน (Grayscale + Normalize 0.5)
โหลดโมเดลด้วย `state_dict` ที่ถูกต้อง และแปลผล index → ตัวอักษรไทยจริง

In [ ]:
# เปลี่ยน path เป็นภาพที่ต้องการทดสอบ
demo_path, _ = samples[test_idx[0]]
result = test_single_image(demo_path, model_name='efficientnet_b3', topk=3, show=True)
print(json.dumps(result, ensure_ascii=False, indent=2))